# Error Handling and Validation



```{contents}
:local:
:depth: 2
```


```{index} Decide What a Failure Means Before Catching It
```



```{index} Decide What a Failure Means Before Catching It
```

## Decide What a Failure Means Before Catching It



An employee enters a delivery quantity. There are at least three different problems the application might encounter: the text is not a number, the number violates a business rule, or a separate resource needed to complete the operation is unavailable. These situations can all prevent success, but they do not call for the same response.

Validation asks whether input satisfies a stated contract. Parsing asks whether text can be interpreted as a value. Exception handling responds to an exceptional transfer of control from an operation. A catch block is not a substitute for specifying which quantities are legal, and a successful parse is not a substitute for checking the domain.

| Input or event | Classification | Appropriate next decision |
| --- | --- | --- |
| Text `twelve` for an integer quantity | Representation failure | Request a whole-number input |
| Integer -2 for packages | Domain failure | Explain that quantity must be nonnegative |
| End of the input stream | No further input | Stop or cancel explicitly |
| A resource operation throws a known recoverable exception | Operation failure | Report context or choose an allowed fallback |

Work through the decisions in that order. First handle a missing input value. Then try to interpret available text. Only after a value exists can the program compare it with the business bounds. If you check a default zero after a failed parse without checking the Boolean success result, you can accidentally treat failed input as a legitimate zero-quantity order.

For related explanation, see the further reading below.[^development-reading]


```{index} Follow Control Through an Exception
```



```{index} Follow Control Through an Exception
```

## Follow Control Through an Exception



A try block marks the work whose failure policy you are handling. When an operation throws, normal execution of that block stops at that point; statements after the throwing operation are not completed. A matching catch handles the exception. A finally block provides cleanup when control leaves the try/catch in ordinary managed execution, including when a handled exception occurs. It does not promise that a killed process or failed machine will run your cleanup code.

Catch only a failure for which you have a meaningful policy. An empty catch that continues to print Success converts a visible failure into misleading behavior. Catching every exception and replacing it with zero may also destroy the distinction between an unavailable answer and a real zero result. File-related examples below preview Chapter 9; their resource APIs will be developed there, rather than assumed as prerequisites for validation.

Guided prediction: quantity zero is permitted, a negative quantity is rejected, and nonnumeric text asks for correction. What should the program report for `0`, `-1`, and `x`? Explain why all three should not share one generic Invalid message.

```{dropdown} Compare the outcomes
Zero is a valid value under this contract. Minus one is interpretable but outside the business domain. The letter x has no integer interpretation. Different messages identify what the user can change, while avoiding a false success result.
```


In [1]:
using System;
string text = "-2";
if (!int.TryParse(text, out int quantity)) Console.WriteLine("Enter a whole number.");
else if (quantity < 0) Console.WriteLine("Quantity must be nonnegative.");
else Console.WriteLine($"Accepted quantity: {quantity}");


Quantity must be nonnegative.


Programs fail in predictable ways: bad input, missing files, unexpected values, and invalid assumptions.
Good software does not pretend errors never happen — it handles them deliberately.

```{index} Validation vs. Exceptions
```



```{index} Validation vs. Exceptions
```

## Validation vs. Exceptions




- **Validation** checks expected bad input *before* doing risky work.
- **Exceptions** handle unexpected failures *at runtime*.

Validation should be your first line of defense. Exceptions are a fallback for when something still goes wrong.

```{index} Basic try / catch / finally
```



```{index} Basic try / catch / finally
```

## Basic `try` / `catch` / `finally`




Use `finally` for cleanup (closing resources, resetting state):

In [ ]:
try
{
    Console.Write("Enter a number: ");
    int value = int.Parse(Console.ReadLine()!);
    Console.WriteLine($"You entered {value}");
}
catch (FormatException)
{
    Console.WriteLine("Input must be a whole number.");
}
catch (OverflowException)
{
    Console.WriteLine("Number is outside the valid int range.");
}
finally
{
    Console.WriteLine("Input attempt complete.");
}

```{index} Prefer TryParse for User Input
```



```{index} Prefer TryParse for User Input
```

## Prefer `TryParse` for User Input




If invalid input is *expected*, avoid throwing exceptions entirely:

In [ ]:
Console.Write("Enter age: ");
string raw = Console.ReadLine() ?? "";

if (int.TryParse(raw, out int age) && age >= 0)
{
    Console.WriteLine($"Age recorded: {age}");
}
else
{
    Console.WriteLine("Please enter a non-negative whole number.");
}

**Guideline:**
- Use `TryParse` for normal input flow where bad input is common.
- Use `try/catch` around operations that can fail unpredictably (file I/O, network, external dependencies).

```{index} File I/O with Safe Handling
```



```{index} File I/O with Safe Handling
```

## File I/O with Safe Handling




Catch specific exception types before broad ones:

In [ ]:
string path = "scores.txt";

try
{
    using StreamReader reader = new StreamReader(path);
    string? line;
    while ((line = reader.ReadLine()) != null)
        Console.WriteLine(line);
}
catch (FileNotFoundException)
{
    Console.WriteLine($"Could not find file: {path}");
}
catch (UnauthorizedAccessException)
{
    Console.WriteLine("Permission denied while reading the file.");
}
catch (IOException ex)
{
    Console.WriteLine($"I/O error: {ex.Message}");
}

```{index} Throwing Exceptions Intentionally
```



```{index} Throwing Exceptions Intentionally
```

## Throwing Exceptions Intentionally




Throw when an API is used incorrectly and execution should stop:

In [ ]:
static double Divide(double numerator, double denominator)
{
    if (denominator == 0)
        throw new ArgumentException("Denominator cannot be zero.", nameof(denominator));

    return numerator / denominator;
}

Console.WriteLine(Divide(10, 2));   // 5
Console.WriteLine(Divide(10, 0));   // throws

This is better than returning meaningless sentinel values that hide bugs.

```{index} Error Message Quality
```



```{index} Error Message Quality
```

## Error Message Quality




| Bad | Better |
|-----|--------|
| `"Error happened"` | `"Could not open 'scores.txt'. Check that the file exists and that you have read permission."` |

Good error messages are: **specific**, **user-actionable**, and free of raw internal noise.

```{index} Common Mistakes
```



```{index} Common Mistakes
```

## Common Mistakes




- Catching `Exception` everywhere and ignoring details
- Swallowing exceptions without logging or reporting
- Using exceptions for normal control flow (use `TryParse` instead)
- Showing stack traces directly to end users

```{index} Practice
```

## Practice




1. Read a value from the console and keep prompting until a valid `double` is entered.
2. Open a file path provided by the user; print a friendly message for: file not found, access denied, and unknown I/O error.
3. Add argument checks to one of your existing methods and throw `ArgumentException` when input is invalid.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [Microsoft, Exceptions and Exception Handling](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/exceptions/). The worked business scenarios and guided reasoning here are original teaching examples.
